In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

In [31]:
m = 6
n = 4

A = [-0.56047565,  0.4609162,  0.4007715,  0.7013559,
    -0.23017749, -1.2650612,  0.1106827, -0.4727914,
    1.55870831, -0.6868529, -0.5558411, -1.0678237,
    0.07050839, -0.4456620,  1.7869131, -0.2179749,
    0.12928774,  1.2240818,  0.4978505, -1.0260044,
    1.71506499,  0.3598138, -1.9666172, -0.7288912]
A = np.array(A).reshape((m,n))
A

array([[-0.56047565,  0.4609162 ,  0.4007715 ,  0.7013559 ],
       [-0.23017749, -1.2650612 ,  0.1106827 , -0.4727914 ],
       [ 1.55870831, -0.6868529 , -0.5558411 , -1.0678237 ],
       [ 0.07050839, -0.445662  ,  1.7869131 , -0.2179749 ],
       [ 0.12928774,  1.2240818 ,  0.4978505 , -1.0260044 ],
       [ 1.71506499,  0.3598138 , -1.9666172 , -0.7288912 ]])

In [32]:
np.random.seed(123)

m = 6
n = 4

A = np.array(stats.norm.rvs(size=m*n)).reshape((m,n))
A

array([[-1.0856306 ,  0.99734545,  0.2829785 , -1.50629471],
       [-0.57860025,  1.65143654, -2.42667924, -0.42891263],
       [ 1.26593626, -0.8667404 , -0.67888615, -0.09470897],
       [ 1.49138963, -0.638902  , -0.44398196, -0.43435128],
       [ 2.20593008,  2.18678609,  1.0040539 ,  0.3861864 ],
       [ 0.73736858,  1.49073203, -0.93583387,  1.17582904]])

In [33]:
AtA = A.T @ A
lambd, v = np.linalg.eig(AtA)
print(lambd, v)

[13.94364275 11.43321631  3.11730374  6.8345766 ] [[-0.58188182 -0.65563318 -0.35493894  0.32492618]
 [-0.76414561  0.44242188  0.00581668 -0.46937249]
 [ 0.16429963 -0.56969869  0.047069   -0.80388652]
 [-0.22472352 -0.2232743   0.93368572  0.16696976]]


In [34]:
sigma = np.sqrt(lambd)
Sigma = np.diag(sigma) 
Sigma

array([[3.73411874, 0.        , 0.        , 0.        ],
       [0.        , 3.38130394, 0.        , 0.        ],
       [0.        , 0.        , 1.76558878, 0.        ],
       [0.        , 0.        , 0.        , 2.61430232]])

In [35]:
U = A @ v @ np.diag(1/sigma)
U

array([[ 0.06817799,  0.39278555, -0.56748918, -0.4972128 ],
       [-0.32874604,  0.76545096, -0.16975489,  0.35038719],
       [-0.04407113, -0.23823594, -0.32553137,  0.51566103],
       [-0.09505201, -0.26929059, -0.54345266,  0.40885177],
       [-0.77031109, -0.33627046, -0.2052653 , -0.40252298],
       [-0.53190363,  0.13210882,  0.45353501,  0.18686204]])

In [36]:
A_reconstruida =  U @ Sigma @ v.T
A_reconstruida

array([[-1.0856306 ,  0.99734545,  0.2829785 , -1.50629471],
       [-0.57860025,  1.65143654, -2.42667924, -0.42891263],
       [ 1.26593626, -0.8667404 , -0.67888615, -0.09470897],
       [ 1.49138963, -0.638902  , -0.44398196, -0.43435128],
       [ 2.20593008,  2.18678609,  1.0040539 ,  0.3861864 ],
       [ 0.73736858,  1.49073203, -0.93583387,  1.17582904]])

In [37]:
error = np.sum((A-A_reconstruida)**2)
error

np.float64(6.825495722908364e-30)

In [38]:
U_svd, D_svd, V_svd = np.linalg.svd(A)
Sigma = np.zeros(A.shape)
Sigma[:n, :n] = np.diag(D_svd)

A_v2 =  U_svd @ Sigma @ V_svd
print(A_v2)
error = np.sum((A-A_v2)**2)
error

[[-1.0856306   0.99734545  0.2829785  -1.50629471]
 [-0.57860025  1.65143654 -2.42667924 -0.42891263]
 [ 1.26593626 -0.8667404  -0.67888615 -0.09470897]
 [ 1.49138963 -0.638902   -0.44398196 -0.43435128]
 [ 2.20593008  2.18678609  1.0040539   0.3861864 ]
 [ 0.73736858  1.49073203 -0.93583387  1.17582904]]


np.float64(9.206137727165426e-30)

In [39]:
u1 = U_svd[:, [0]] 
v1_t = V_svd[[0], :] 
sigma1 = D_svd[0]

A1 = sigma1 * (u1 @ v1_t)
error_rango1 = np.sum((A - A1) ** 2)
error_rango1

np.float64(21.385096653186295)

In [40]:
k = 2
Uk = U_svd[:, :k]
Vk = V_svd[:k, :]
Sigmak = np.diag(D_svd[:k])

Ak = Uk @ Sigmak @ Vk
error_rango_k = np.sum((A - Ak) ** 2)
print(error_rango_k)

9.951880340837572


In [41]:
total = np.sum(D_svd**2)
var_exp = D_svd**2 / total * 100
var_cum = np.cumsum(var_exp)

pd.DataFrame({'Componente': [i for i in range(1, len(D_svd)+1)],
            'Varianza': var_exp,
            'Acumulada': var_cum})

,Componente,Varianza,Acumulada
0,1,39.468271,39.468271
1,2,32.362367,71.830638
2,3,19.345657,91.176295
3,4,8.823705,100.000000
